# Molecular Generator Evaluation using RS, SED and ASER Pharmacophore-Based Metrics

### 🔹 Objective
This notebook evaluates molecular generators by computing key metrics using **pharmacophore fingerprints**:

- **RS_pharm**: pharmacophore fingerprint recall metrics  
- **SED_pharm**: pharmacophore hopping potential  
- **ASER_pharm**: chemical space exploration based on pharmacophore features  

### 🔹 Workflow
1️⃣ **Compute Pharmacophore Fingerprints**  
> Before metric calculation, all necessary pharmacophore fingerprints for the molecules must be precomputed due to computational complexity.
> This notebook does not generate pharmacophore fingerprints. It expects precomputed input files in `data/output_sets/ph4/...`.
> A dedicated preprocessing script for pharmacophore fingerprint generation should be run before this notebook.

2️⃣ **Compute Metrics**  
> The script calculates RS, SED, and ASER for different molecular generators using pharmacophore fingerprints.

3️⃣ **Merge Data**  
> Results from multiple generators are combined into a single Pandas DataFrame.

4️⃣ **Normalize Values**  
> The computed metrics are normalized using Min-Max scaling for comparison.

5️⃣ **Save Outputs**  
> Processed data is stored in CSV files for further analysis.

### 🔹 Data Structure
- Calculations are performed for **RDKit pharmacophore fingerprint type** and **cluster types** (`dis`, `sim`).  
- Results are computed for multiple **generators** (`Molpher`, `REINVENT`, `DrugEx`, `GB_GA`, `AddCarbon`).  
- The analysis is conducted for a specific **biological target receptor**, such as the **Glucocorticoid receptor**.

### Hints:
Recommendation thresholds: for Dissimilarity split threshold = 0.7/ For Similarity split threshold = 0.8. Set `DATA_FOLDER` to the project root containing the `data/` directory.

Expected precomputed files:
- `data/output_sets/ph4/{receptor}/{generator}/phfp_of_output_set_cluster_{n}_{split}_{generator}_with_smiles.csv`
- `data/output_sets/ph4/{receptor}/RS/phfp_of_recall_set_cluster_{n}_{split}_with_smiles.csv`


# Loading required libraries

In [1]:
from src import metrics_ph4 as mph4 # Importing custom metric functions
import importlib as imp
imp.reload(mph4)

<module 'src.metrics_ph4' from '/home/filv/phd_projects/iga_2023/git_reccal/new/diseration_git/generative_models_for_de_novo_molecular_design/src/metrics_ph4.py'>

In [2]:
DATA_FOLDER = '../../'  # project root containing the data/ directory; results are saved there as well
DEFAULT_NCPUS = 8
THRESHOLD_DIS = 0.7
THRESHOLD_SIM = 0.8
# Prerequisite: pharmacophore fingerprint CSV files must already exist in data/output_sets/ph4/.

# Optional: precompute pharmacophore fingerprints

If the fingerprint CSV files are not available yet, you can generate them before metric calculation using `src/compute_pharmacophore_fingerprints.py`.

The script reads:
- output set SMILES from `data/output_sets/{receptor}/{generator}/cOS_..._one_column.csv`
- recall set SMILES from `data/input_recall_sets/{receptor}/cRS_...csv`
- optional input set SMILES from `data/input_recall_sets/{receptor}/cIS_...csv`

and writes pharmacophore fingerprint CSV files into `data/output_sets/ph4/...`.

In [ ]:
import subprocess
import sys

RUN_PHFP_PRECOMPUTE = False
PRECOMPUTE_RECEPTOR = 'Leukocyte_elastase'
PRECOMPUTE_GENERATOR = 'Molpher'
PRECOMPUTE_SPLIT = 'sim'
PRECOMPUTE_CLUSTERS = [0, 1, 2, 3, 4]
PRECOMPUTE_NCPUS = DEFAULT_NCPUS

if RUN_PHFP_PRECOMPUTE:
    command = [
        sys.executable,
        'src/compute_pharmacophore_fingerprints.py',
        '--receptor', PRECOMPUTE_RECEPTOR,
        '--generator', PRECOMPUTE_GENERATOR,
        '--split', PRECOMPUTE_SPLIT,
        '--type_phfp', 'rdkit',
        '--dataset', 'all',
        '--ncpus', str(PRECOMPUTE_NCPUS),
        '--data_folder', DATA_FOLDER,
        '--clusters', *map(str, PRECOMPUTE_CLUSTERS),
    ]
    subprocess.run(command, check=True)
else:
    print('Fingerprint precomputation skipped. Set RUN_PHFP_PRECOMPUTE = True to generate them.')


# Function to calculate metrics

In [3]:
def calculate_metrics(type_cluster, type_phfp, generator, receptor, threshold=1, data_folder='.', ncpus=1):
    """
    Calculate pharmacophore-based metrics.
    
    Parameters:
    - type_phfp: Type of pharmacophore fingeprint (e.g., 'rdkit')
    - type_cluster: Cluster type  (e.g., 'dis' or 'sim') dis = Dissimilarity split; sim = Similarity split
    - generator: Name of the molecular generator with subset 250k (e.g. 'Molpher_250k', 'REINVENT_250k', 'DrugEx_GT_epsilon_0.1', 'DrugEx_GT_epsilon_0.6'
        'DrugEx_RNN_epsilon_0.1_250k', 'DrugEx_RNN_epsilon_0.6_250k', 'GB_GA_mut_r_0.01_250k', 'GB_GA_mut_r_0.5_250k') without subset: 'GB_GA_log_p_mut_r_0.01','GB_GA_log_p_mut_r_0.5')
    - receptor: Target receptor for drug design (e.g. 'Glucocorticoid_receptor', 'Leukocyte_elastase')
    - threshold: Tanimoto similarity threshold.
    - data_folder: Path to the project root containing the 'data/' directory. Results are saved there as well.
    - ncpus: Number of CPUs used for similarity calculations.

    Returns:
    - Computed metrics
    """
    mt = mph4.MetricsPh4(type_cluster, type_phfp, generator, receptor, threshold, data_folder, ncpus)
    result = mt.calculate()
    display(result)
    return result

# Define parameters for metric calculations

In [ ]:
type_cluster = 'sim' #options: 'dis'|'sim' 
type_phfp = 'rdkit' #options: 'rdkits'
generator = 'GB_GA_log_p_mut_r_0.01' #options: 'Molpher'|'DrugEx_GT_epsilon_0.1'|'REINVENT'|'GB_GA_mut_r_0.5'|'GB_GA_log_p_mut_r_0.01'
receptor = 'Leukocyte_elastase' #options: 'Glucocorticoid_receptor'|'Leukocyte_elastase' 
data_folder = DATA_FOLDER
threshold = THRESHOLD_SIM
calculate_metrics(type_cluster, type_phfp, generator, receptor, threshold, data_folder, ncpus=DEFAULT_NCPUS)


Starting calculation for GB_GA_log_p_mut_r_0.01
Receptor: Leukocyte_elastase
Cluster type: sim
Threshold: 0.8

--- Processing cluster 0 ---

Loading data from:
  Output: /home/filv/phd_projects/iga_2023/git_reccal/new/data/output_sets/ph4/Leukocyte_elastase/GB_GA_log_p_mut_r_0.01/phfp_of_output_set_cluster_0_sim_GB_GA_log_p_mut_r_0.01_with_smiles.csv
  Recall: /home/filv/phd_projects/iga_2023/git_reccal/new/data/output_sets/ph4/Leukocyte_elastase/RS/phfp_of_recall_set_cluster_0_sim_with_smiles.csv
Original output length: 1,000,000
Original recall length: 644
Unique recall length: 349

Calculating matching statistics...
Converting recall fingerprints...
  Converting 349 fingerprints...
Converting output fingerprints...
  Converting 1,000,000 fingerprints...
    Progress: 50,000/1,000,000 (5.0%)
    Progress: 100,000/1,000,000 (10.0%)
    Progress: 150,000/1,000,000 (15.0%)
    Progress: 200,000/1,000,000 (20.0%)
    Progress: 250,000/1,000,000 (25.0%)


In [ ]:
type_phfp = 'rdkit' #options: 'rdkit'
for type_cluster in ['dis']: #options: 'dis'|'sim' 
    for receptor in ['Glucocorticoid_receptor', 'Leukocyte_elastase']:
        for generator in [ 'DrugEx_GT_epsilon_0.6', 'DrugEx_RNN_epsilon_0.1','DrugEx_RNN_epsilon_0.6', 'GB_GA_mut_r_0.01',\
                          'GB_GA_mut_r_0.5', 'GB_GA_log_p_mut_r_0.01', 'GB_GA_log_p_mut_r_0.5', 'enamine', 'Molpher',\
                          'REINVENT', 'DrugEx_GT_epsilon_0.1']:
            data_folder = DATA_FOLDER
            threshold = THRESHOLD_DIS
            calculate_metrics(type_cluster, type_phfp, generator, receptor, threshold, data_folder, ncpus=DEFAULT_NCPUS)

## Combining and Normalizing Metrics

The following cell runs functions that:

- merge the mean values of all metrics into a single `pandas.DataFrame` (using `connect_mean_value`)
- apply Min-Max normalization to scale the values (using `connect_mean_value_normalized`)


In [31]:
from src import metrics_connection_phfp # Importing custom metric functions
imp.reload(metrics_connection_phfp)

<module 'src.metrics_connection_phfp' from '/home/filv/phd_projects/iga_2023/git_reccal/new/diseration_git/generative_models_for_de_novo_molecular_design/src/metrics_connection_phfp.py'>

In [33]:
data_folder = DATA_FOLDER
threshold_sim = THRESHOLD_SIM
threshold_dis = THRESHOLD_DIS
type_phfp = 'rdkit'

for receptor in ['Glucocorticoid_receptor','Leukocyte_elastase']:
    for type_cluster in ['dis','sim']:  # Different cluster types
            threshold = threshold_sim if type_cluster == "sim" else threshold_dis
            # Define generator names with different epsilon values
            generator_list = [
                f"Molpher",
                f"REINVENT",
                f"DrugEx_GT_epsilon_0.1",
                f"DrugEx_GT_epsilon_0.6",
                f"DrugEx_RNN_epsilon_0.1",
                f"DrugEx_RNN_epsilon_0.6",
                f"GB_GA_mut_r_0.01",
                f"GB_GA_mut_r_0.5",
                f"GB_GA_log_p_mut_r_0.01",
                f"GB_GA_log_p_mut_r_0.5",
                f"enamine"
            ]
            
            df_raw = metrics_connection_phfp.connect_mean_value( type_cluster, type_phfp, generator_list, receptor, threshold, data_folder)
            display(df_raw[['name','type_cluster','phfp','RS','SED','ASER']])
            out_xlsx = (
                f"../data/results_pharm_based/{receptor}/{type_phfp}/{type_cluster}/"
                f"paper_table_{receptor}_{type_phfp}_{type_cluster}_threshold_{threshold}.xlsx"
            )
            metrics_connection_phfp.export_article_excel_table(
                df=df_raw,
                out_xlsx=out_xlsx,
                receptor=receptor,
                type_phfp=type_phfp,
                threshold=threshold,
                type_cluster=type_cluster, 
            )

[OK] Excel saved to: ../data/results_pharm_fp/Glucocorticoid_receptor/rdkit/dis/paper_table_Glucocorticoid_receptor_rdkit_dis_threshold_0.7.xlsx
[OK] Excel saved to: ../data/results_pharm_fp/Glucocorticoid_receptor/rdkit/sim/paper_table_Glucocorticoid_receptor_rdkit_sim_threshold_0.8.xlsx
[OK] Excel saved to: ../data/results_pharm_fp/Leukocyte_elastase/rdkit/dis/paper_table_Leukocyte_elastase_rdkit_dis_threshold_0.7.xlsx
[OK] Excel saved to: ../data/results_pharm_fp/Leukocyte_elastase/rdkit/sim/paper_table_Leukocyte_elastase_rdkit_sim_threshold_0.8.xlsx
